# 函数式 API 概念
## 概述¶
功能API允许您将 LangGraph 的主要功能（[持久性](https://langchain-ai.github.io/langgraph/concepts/persistence/)、[内存](https://langchain-ai.github.io/langgraph/how-tos/memory/add-memory/)、[人在环](https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/)和[流式传输](https://langchain-ai.github.io/langgraph/concepts/streaming/)）添加到您的应用程序中，而只需对现有代码进行最少的更改。


它旨在将这些功能集成到现有代码中，这些代码可能使用标准语言原语（例如if语句、for循环和函数调用）来实现分支和控制流。与许多需要将代码重构为显式流水线或 DAG 的数据编排框架不同，Functional API 允许您在不强制执行严格执行模型的情况下整合这些功能。

Functional API 使用两个关键构建块：
- `@entrypoint`– 将某个函数标记为工作流的起点，封装逻辑并管理执行流程，包括处理长时间运行的任务和中断。
- `@task`– 表示一个独立的工作单元，例如 API 调用或数据处理步骤，可以在入口点内异步执行。任务返回一个类似 Future 的对象，可以等待或同步执行。

这为构建具有状态管理和流式传输的工作流提供了最小的抽象。

> 有关如何使用功能 API 的信息，请参[阅使用功能 API](https://langchain-ai.github.io/langgraph/how-tos/use-functional-api/)。


## 函数式 API 与图形 API¶
对于更喜欢声明式方法的用户，LangGraph 的Graph API允许您使用 Graph 范式定义工作流。这两个 API 共享相同的底层运行时，因此您可以在同一个应用程序中一起使用它们。

以下是一些主要区别：

- 控制流：函数式 API 无需考虑图结构。您可以使用标准的 Python 结构来定义工作流。这通常会减少您需要编写的代码量。
- 短期记忆：GraphAPI需要声明一个状态，并且可能需要定义减速器来管理图形状态的更新。@entrypoint并且@tasks不需要明确的状态管理，因为它们的状态范围限定于函数，并且不会在函数之间共享。
- 检查点：两种 API 都会生成并使用检查点。在Graph API中，每个超级步骤之后都会生成一个新的检查点。在Functional API中，执行任务时，其结果会保存到与给定入口点关联的现有检查点中，而不是创建新的检查点。
- 可视化：Graph API 可以轻松地将工作流可视化为图形，这有助于调试、理解工作流以及与他人共享。Functional API 不支持可视化，因为图形是在运行时动态生成的。


## 例子¶
下面我们演示一个简单的应用程序，它可以写一篇文章并打断请求人工审核。




In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langgraph.types import interrupt
import time

@task
def write_essay(topic: str) -> str:
    """Write an essay about the given topic."""
    time.sleep(1) # A placeholder for a long-running task.
    return f"An essay about topic: {topic}"

@entrypoint(checkpointer=InMemorySaver())
def workflow(topic: str) -> dict:
    """A simple workflow that writes an essay and asks for a review."""
    essay = write_essay("cat").result()
    is_approved = interrupt({
        # Any json-serializable payload provided to interrupt as argument.
        # It will be surfaced on the client side as an Interrupt when streaming data
        # from the workflow.
        "essay": essay, # The essay we want reviewed.
        # We can add any additional information that we need.
        # For example, introduce a key called "action" with some instructions.
        "action": "Please approve/reject the essay",
    })

    return {
        "essay": essay, # The essay that was generated
        "is_approved": is_approved, # Response from HIL
    }

## 入口点¶
装饰[@entrypoint](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.entrypoint)器可用于从函数创建工作流。它封装了工作流逻辑并管理执行流程，包括处理长时间运行的任务和[中断](https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/)。

### 定义¶
入口点是通过使用`@entrypoint`装饰器来装饰函数来定义的。

该函数必须接受单个位置参数，作为工作流的输入。如果需要传递多条数据，请使用字典作为第一个参数的输入类型。


使用装饰函数`entrypoint`可以生成一个`Pregel`实例，帮助管理工作流的执行（例如，处理流、恢复和检查点）。

您通常需要将检查点传递给`@entrypoint`装饰器以实现持久性并使用诸如人机交互之类的功能。



In [ ]:
## 同步

from langgraph.func import entrypoint

@entrypoint(checkpointer=checkpointer)
def my_workflow(some_input: dict) -> int:
    # some logic that may involve long-running tasks like API calls,
    # and may be interrupted for human-in-the-loop.
    ...
    return result

In [ ]:
## 异步
from langgraph.func import entrypoint

@entrypoint(checkpointer=checkpointer)
async def my_workflow(some_input: dict) -> int:
    # some logic that may involve long-running tasks like API calls,
    # and may be interrupted for human-in-the-loop
    ...
    return result 

> 入口点的输入和输出必须支持JSON序列化才能支持检查点。更多详情，请参阅序列化部分

#### 可注入参数¶
声明`entrypoint`时，您可以请求访问将在运行时自动注入的其他参数。这些参数包括：

| 参数 | 描述 |
| :--- | :--- |
| previous | 访问与给定线程的上一个检查点相关的状态。参见[短记忆](https://langchain-ai.github.io/langgraph/concepts/functional_api/#short-term-memory)。 |
| store | BaseStore 的一个[实例](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.BaseStore)。可用于[长记忆](https://langchain-ai.github.io/langgraph/how-tos/use-functional-api/#long-term-memory)。 |
| writer | 用于在使用 Async Python < 3.11 时访问 StreamWriter。详情请参阅使用函数式 API 进行[流式处理](https://langchain-ai.github.io/langgraph/how-tos/use-functional-api/#streaming)。 |
| config | 用于访问运行时配置。有关信息，请参阅 [RunnableConfig](https://python.langchain.com/docs/concepts/runnables/?_gl=1*bgntqe*_gcl_au*NDE1NjY4Mjc0LjE3NTM0Mjc3MTc.*_ga*MTEyMjY1OTA5MS4xNzUzNDI3NzE4*_ga_47WX3HKKY2*czE3NTM2ODMxOTYkbzkkZzEkdDE3NTM2ODMyMjIkajM0JGwwJGgw#runnableconfig)。 |


#### 执行 
使用`@entrypoint`会产生一个`Pregel`可以使用`invoke`、`ainvoke`和`streamastream`方法执行的对象。



In [ ]:
config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}
my_workflow.invoke(some_input, config)  # Wait for the result synchronously



In [ ]:
config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}
await my_workflow.ainvoke(some_input, config)  # Await result asynchronously

In [ ]:
config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}

for chunk in my_workflow.stream(some_input, config):
    print(chunk)

In [ ]:
config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}

async for chunk in my_workflow.astream(some_input, config):
    print(chunk)

#### 恢复¶
可以通过将恢复值传递给命令原语来在中断后恢复执行。


In [ ]:
from langgraph.types import Command

config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}

my_workflow.invoke(Command(resume=some_resume_value), config)

In [ ]:
from langgraph.types import Command

config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}

await my_workflow.ainvoke(Command(resume=some_resume_value), config)

In [ ]:
from langgraph.types import Command

config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}

for chunk in my_workflow.stream(Command(resume=some_resume_value), config):
    print(chunk)



In [ ]:
from langgraph.types import Command

config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}

async for chunk in my_workflow.astream(Command(resume=some_resume_value), config):
    print(chunk)

#### 发生错误后恢复

要在错误发生后恢复，请`entrypoint`使用None相同的线程 ID（配置）运行。

这假设底层错误已经解决并且可以成功执行。

In [ ]:
config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}

my_workflow.invoke(None, config)

In [ ]:
config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}

await my_workflow.ainvoke(None, config)

In [ ]:
config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}

for chunk in my_workflow.stream(None, config):
    print(chunk)

In [ ]:
config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}

async for chunk in my_workflow.astream(None, config):
    print(chunk)

#### 短期记忆¶
当用`entrypoint`定义时，它会在检查点checkpointer中存储同一线程 ID上的连续调用之间的信息。

这允许使用参数访问前一次调用previous的状态。

默认情况下，该previous参数是前一次调用的返回值。



In [ ]:
@entrypoint(checkpointer=checkpointer)
def my_workflow(number: int, *, previous: Any = None) -> int:
    previous = previous or 0
    return number + previous

config = {
    "configurable": {
        "thread_id": "some_thread_id"
    }
}

my_workflow.invoke(1, config)  # 1 (previous was None)
my_workflow.invoke(2, config)  # 3 (previous was 1 from the previous invocation)

##### entrypoint.final¶
`entrypoint.final`是一个特殊的原语，可以从入口点返回，并允许将检查点中保存的值与入口点的返回值分离。

第一个值是入口点的返回值，第二个值是将在检查点中保存的值。类型注释为`entrypoint.final[return_type, save_type]`。

In [ ]:
@entrypoint(checkpointer=checkpointer)
def my_workflow(number: int, *, previous: Any = None) -> entrypoint.final[int, int]:
    previous = previous or 0
    # This will return the previous value to the caller, saving
    # 2 * number to the checkpoint, which will be used in the next invocation 
    # for the `previous` parameter.
    return entrypoint.final(value=previous, save=2 * number)

config = {
    "configurable": {
        "thread_id": "1"
    }
}

my_workflow.invoke(3, config)  # 0 (previous was None)
my_workflow.invoke(1, config)  # 6 (previous was 3 * 2 from the previous invocation)

### 任务¶
任务代表一个独立的工作单元，例如一次 API 调用或数据处理步骤。它具有两个关键特征：

- 异步执行：任务被设计为异步执行，允许多个操作同时运行而不会阻塞。
- 检查点：任务结果将保存到检查点，以便从上次保存的状态恢复工作流程。（有关更多详细信息，请参阅持久性）。

#### 定义¶
任务是使用`@task`装饰器定义的，它包装了常规的 Python 函数。

API 参考：[任务](https://langchain-ai.github.io/langgraph/reference/func/#langgraph.func.task)

In [ ]:
from langgraph.func import task

@task()
def slow_computation(input_value):
    # Simulate a long-running operation
    ...
    return result

> 任务的输出必须是 JSON 可序列化才能支持检查点。

#### 执行¶
任务只能从入口点、另一个任务或状态图节点内调用。

不能从主应用程序代码直接从调用任务。

当你调用一个task时，它会立即返回一个 Future 对象。Future 是一个占位符，用于保存稍后可用的结果。

要获得任务的结果，您可以同步等待它（使用result()）或异步等待它（使用await）。

In [ ]:
@entrypoint(checkpointer=checkpointer)
def my_workflow(some_input: int) -> int:
    future = slow_computation(some_input)
    return future.result()  # Wait for the result synchronously

In [ ]:
@entrypoint(checkpointer=checkpointer)
async def my_workflow(some_input: int) -> int:
    return await slow_computation(some_input)  # Await result asynchronously

## 何时使用任务¶
任务在以下场景中很有用：

- 检查点：当您需要将长时间运行的操作的结果保存到检查点时，您不需要在恢复工作流时重新计算它。
- 人机交互：如果您正在构建需要人工干预的工作流，则必须使用任务来封装任何随机性（例如 API 调用），以确保工作流能够正确恢复。更多详情，请参阅确定性部分。
- 并行执行：对于 I/O 密集型任务，任务支持并行执行，允许多个操作同时运行而不会阻塞（例如，调用多个 API）。
- 可观察性：将操作包装在任务中提供了一种使用LangSmith跟踪工作流进度和监视各个操作执行情况的方法。
- 可重试工作：当需要重试工作以处理故障或不一致时，任务提供了一种封装和管理重试逻辑的方法。

# 序列化¶
LangGraph 中的序列化有两个关键方面：

- `@entrypoin`t输入和输出必须是 JSON 可序列化的。
- `@task`输出必须是 JSON 可序列化的。

这些要求对于启用检查点和工作流恢复至关重要。使用 Python 原语（例如字典、列表、字符串、数字和布尔值）来确保输入和输出可序列化。

序列化确保工作流状态（例如任务结果和中间值）能够可靠地保存和恢复。这对于实现人机交互、容错和并行执行至关重要。

当工作流配置了检查点时，提供不可序列化的输入或输出将导致运行时错误。

## 决定论¶
为了利用诸如“人机循环”之类的功能，任何随机性都应封装在任务中。这可以保证当执行暂停（例如，对于“人机循环”）并恢复时，即使任务结果不确定，它也将遵循相同的步骤顺序。

LangGraph 通过在任务和子图执行过程中持久化结果来实现此行为。精心设计的工作流程可确保恢复执行遵循相同的步骤顺序，从而允许正确检索先前计算的结果，而无需重新执行。这对于长时间运行的任务或结果不确定的任务尤其有用，因为它避免了重复之前完成的工作，并允许从基本相同的位置恢复。

虽然工作流的不同运行可能会产生不同的结果，但恢复特定运行应始终遵循相同的记录步骤顺序。这使得 LangGraph 能够高效地查找在图表中断之前执行的任务和子图结果，并避免重新计算它们。

## 幂等性¶
幂等性确保多次运行相同的操作会产生相同的结果。这有助于避免因故障而重新运行某个步骤时出现重复的 API 调用和冗余处理。始终将 API 调用置于任务函数内部以进行检查点设置，并将其设计为在重新执行时具有幂等性。如果任务启动但未成功完成，则可能会重新执行。然后，如果工作流程恢复，该任务将再次运行。使用幂等性键或验证现有结果以避免重复。

## 常见陷阱¶
### 处理副作用¶
将副作用（例如，写入文件、发送电子邮件）封装在任务中，以确保在恢复工作流时不会多次执行它们。

在这个例子中，副作用（写入文件）直接包含在工作流中，因此在恢复工作流时它将被第二次执行。

In [ ]:
@entrypoint(checkpointer=checkpointer)
def my_workflow(inputs: dict) -> int:
    # This code will be executed a second time when resuming the workflow.
    # Which is likely not what you want.
    with open("output.txt", "w") as f:
        f.write("Side effect executed")
    value = interrupt("question")
    return value

在这个例子中，副作用被封装在一个任务中，确保恢复时执行的一致性。

In [ ]:
@task
def write_to_file():
    with open("output.txt", "w") as f:
        f.write("Side effect executed")

@entrypoint(checkpointer=checkpointer)
def my_workflow(inputs: dict) -> int:
    # The side effect is now encapsulated in a task.
    write_to_file().result()
    value = interrupt("question")
    return value

## 非确定性控制流¶
每次可能产生不同结果的操作（如获取当前时间或随机数）应封装在任务中，以确保在恢复时返回相同的结果。

- 在一个任务中：获取随机数（5）→中断→恢复→（再次返回5）→...
- 不在任务中：获取随机数（5）→中断→恢复→获取新的随机数（7）→...

当使用包含多个中断调用的人机交互工作流时，这一点尤为重要。LangGraph 为每个任务/入口点保存一个恢复值列表。当遇到中断时，它会与相应的恢复值进行匹配。这种匹配严格基于索引，因此恢复值的顺序应与中断的顺序匹配。

如果在恢复时没有保持执行顺序，则一个interrupt调用可能会与错误的resume值匹配，从而导致不正确的结果。

请阅读有关确定性的部分以了解更多详细信息。

在此示例中，工作流使用当前时间来确定要执行哪个任务。这是不确定的，因为工作流的结果取决于其执行的时间。

In [ ]:
from langgraph.func import entrypoint

@entrypoint(checkpointer=checkpointer)
def my_workflow(inputs: dict) -> int:
    t0 = inputs["t0"]
    t1 = time.time()

    delta_t = t1 - t0

    if delta_t > 1:
        result = slow_task(1).result()
        value = interrupt("question")
    else:
        result = slow_task(2).result()
        value = interrupt("question")

    return {
        "result": result,
        "value": value
    }

在此示例中，工作流使用输入t0来确定要执行哪个任务。这是确定性的，因为工作流的结果仅取决于输入。

In [ ]:
import time

from langgraph.func import task

@task
def get_time() -> float:
    return time.time()

@entrypoint(checkpointer=checkpointer)
def my_workflow(inputs: dict) -> int:
    t0 = inputs["t0"]
    t1 = get_time().result()

    delta_t = t1 - t0

    if delta_t > 1:
        result = slow_task(1).result()
        value = interrupt("question")
    else:
        result = slow_task(2).result()
        value = interrupt("question")

    return {
        "result": result,
        "value": value
    }